In [7]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import BaseMessage, HumanMessage
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
import operator

In [8]:
model = ChatOllama(model="phi3:mini")

In [9]:
class ChatState(TypedDict):
    chat:Annotated[list[BaseMessage], add_messages]

In [10]:
def chat_node(state : ChatState):
    chat = state['chat']

    response = model.invoke(chat)

    return {'chat' : [response]}

In [11]:
graph = StateGraph(ChatState)
checkpoint = InMemorySaver()

graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer = checkpoint)

In [12]:
thread_id = '-1'
config1 = {'configurable':{'thread_id':thread_id}}

while True:
    user_input = input("Enter message : ")
    print(f'You : {user_input}\n')

    initial_state = {'chat': [HumanMessage(content = user_input)]}

    final_state = chatbot.invoke(initial_state, config=config1)

    print(f'AI : {final_state['chat'][-1].content}\n')

You : hi my name in muhtasham

AI : Hello! It seems like you're introducing yourself, but it looks like there's a typo in your name. Assuming you wanted to say "my name is Muhtasham," here's a polite response:

Hi there! It's a pleasure to meet you. I'm an AI here to help with any questions or tasks you may have. Thank you for sharing your name; Muhtasham is a beautiful name. How can I assist you today?

You : can you tell me my name

AI : Certainly! Your name, as you mentioned before, is Muhtasham. If there's more you'd like to learn about yourself or any other topic, feel free to ask!



KeyboardInterrupt: Interrupted by user